# Comparaison de prompts — imports

Charge les outils nécessaires à l’évaluation.

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.llm.model import load_model
from src.agents.baseline import BaselineAgent
from src.llm.prompts import (
    build_system_prompt_v1,
    build_system_prompt_v2,
)

# Chargement du modèle

Initialise le modèle utilisé pour tous les tests.

In [2]:
llm = load_model()

Chargement du modèle : Qwen/Qwen2.5-0.5B-Instruct
Device utilisé : cuda


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

# Règles communes

Définit les règles appliquées aux deux prompts.

In [3]:
rules = """
Une personne est admissible si elle a au moins 18 ans.
Une personne de moins de 18 ans n'est pas admissible.
Si l'âge n'est pas connu, il est impossible de prendre une décision.
"""

# Cas de test

Prépare les mêmes exemples pour V1 et V2.

In [5]:
test_cases = [
    {
        "id": 1,
        "fiche": "Nom : Alice\nÂge : 22 ans",
        "question": "Alice est-elle admissible ?",
        "expected": "ADMISSIBLE",
    },
    {
        "id": 2,
        "fiche": "Nom : Bob\nÂge : 16 ans",
        "question": "Bob est-il admissible ?",
        "expected": "NON_ADMISSIBLE",
    },
    {
        "id": 3,
        "fiche": "Nom : Chloé\nÂge : 18 ans",
        "question": "Chloé est-elle admissible ?",
        "expected": "ADMISSIBLE",
    },
    {
        "id": 4,
        "fiche": "Nom : David\nÂge : inconnu",
        "question": "David est-il admissible ?",
        "expected": "INDETERMINE",
    },
    {
        "id": 5,
        "fiche": "Nom : Emma\nÂge : 17 ans",
        "question": "Emma est-elle admissible ?",
        "expected": "NON_ADMISSIBLE",
    },
    {
        "id": 6,
        "fiche": "Nom : Farid\nÂge : 35 ans",
        "question": "Farid est-il admissible ?",
        "expected": "ADMISSIBLE",
    },
]

# Fonction d’évaluation

Exécute un prompt sur tous les cas de test.

In [6]:
def evaluate_prompt(prompt_builder, prompt_name):
    results = []

    for case in test_cases:

        system_prompt = prompt_builder(
            rules=rules,
            fiche=case["fiche"],
        )

        agent = BaselineAgent(
            model=llm.model,
            tokenizer=llm.tokenizer,
            device=llm.device,
            rules=rules,
            fiche=case["fiche"],
            system_prompt=system_prompt,
        )

        result = agent.ask(case["question"])

        predicted = result["verdict"]

        results.append({
            "prompt": prompt_name,
            "case_id": case["id"],
            "expected": case["expected"],
            "predicted": predicted,
            "format_valid": predicted is not None,
            "correct": predicted == case["expected"],
            "response": result["response"],
        })

    return results

# Exécution V1 / V2

Lance les deux variantes et regroupe les résultats.

In [7]:
results_v1 = evaluate_prompt(
    build_system_prompt_v1,
    "V1_simple",
)

results_v2 = evaluate_prompt(
    build_system_prompt_v2,
    "V2_strict",
)

results = results_v1 + results_v2

df = pd.DataFrame(results)

df

,prompt,case_id,expected,predicted,format_valid,correct,response
0,V1_simple,1,ADMISSIBLE,NaN,False,False,"Oui, Alice est admissible car elle a au moins ..."
1,V1_simple,2,NON_ADMISSIBLE,NaN,False,False,"Non, Bob n'est pas admissible car son âge est ..."
2,V1_simple,3,ADMISSIBLE,NaN,False,False,"Oui, Chloé est admissible car elle a au moins ..."
3,V1_simple,4,INDETERMINE,NaN,False,False,"Non, David n'est pas admissible car son âge es..."
4,V1_simple,5,NON_ADMISSIBLE,NaN,False,False,"Oui, Emma est admissible car elle a au moins 1..."
5,V1_simple,6,ADMISSIBLE,NaN,False,False,"Non, Farid n'est pas admissible car son âge es..."
6,V2_strict,1,ADMISSIBLE,ADMISSIBLE,True,True,[VERDICT: ADMISSIBLE]
7,V2_strict,2,NON_ADMISSIBLE,ADMISSIBLE,True,False,[VERDICT: ADMISSIBLE]
8,V2_strict,3,ADMISSIBLE,ADMISSIBLE,True,True,[VERDICT: ADMISSIBLE]
9,V2_strict,4,INDETERMINE,NON_ADMISSIBLE,True,False,[VERDICT: NON_ADMISSIBLE]


# Métriques automatiques

Calcule exactitude brute et respect du format.

In [8]:
summary = (
    df.groupby("prompt")
    .agg(
        accuracy=("correct", "mean"),
        format_rate=("format_valid", "mean"),
        n_cases=("case_id", "count"),
    )
    .reset_index()
)

summary["accuracy"] *= 100
summary["format_rate"] *= 100

summary

,prompt,accuracy,format_rate,n_cases
0,V1_simple,0.0,0.0,6
1,V2_strict,50.0,100.0,6


# Analyse des erreurs

Affiche uniquement les prédictions incorrectes.

In [9]:
df[
    df["correct"] == False
][
    [
        "prompt",
        "case_id",
        "expected",
        "predicted",
        "response",
    ]
]

,prompt,case_id,expected,predicted,response
0,V1_simple,1,ADMISSIBLE,NaN,"Oui, Alice est admissible car elle a au moins ..."
1,V1_simple,2,NON_ADMISSIBLE,NaN,"Non, Bob n'est pas admissible car son âge est ..."
2,V1_simple,3,ADMISSIBLE,NaN,"Oui, Chloé est admissible car elle a au moins ..."
3,V1_simple,4,INDETERMINE,NaN,"Non, David n'est pas admissible car son âge es..."
4,V1_simple,5,NON_ADMISSIBLE,NaN,"Oui, Emma est admissible car elle a au moins 1..."
5,V1_simple,6,ADMISSIBLE,NaN,"Non, Farid n'est pas admissible car son âge es..."
7,V2_strict,2,NON_ADMISSIBLE,ADMISSIBLE,[VERDICT: ADMISSIBLE]
9,V2_strict,4,INDETERMINE,NON_ADMISSIBLE,[VERDICT: NON_ADMISSIBLE]
10,V2_strict,5,NON_ADMISSIBLE,ADMISSIBLE,[VERDICT: ADMISSIBLE]


# Comparaison par cas

Met les résultats V1 et V2 côte à côte.

In [10]:
comparison = df.pivot(
    index="case_id",
    columns="prompt",
    values=["expected", "predicted", "correct"],
)

comparison

expected                 predicted                   correct  \
prompt        V1_simple       V2_strict V1_simple       V2_strict V1_simple   
case_id                                                                       
1            ADMISSIBLE      ADMISSIBLE       NaN      ADMISSIBLE     False   
2        NON_ADMISSIBLE  NON_ADMISSIBLE       NaN      ADMISSIBLE     False   
3            ADMISSIBLE      ADMISSIBLE       NaN      ADMISSIBLE     False   
4           INDETERMINE     INDETERMINE       NaN  NON_ADMISSIBLE     False   
5        NON_ADMISSIBLE  NON_ADMISSIBLE       NaN      ADMISSIBLE     False   
6            ADMISSIBLE      ADMISSIBLE       NaN      ADMISSIBLE     False   

                   
prompt  V2_strict  
case_id            
1            True  
2           False  
3            True  
4           False  
5           False  
6            True

# Test de stabilité

Répète une même requête avec le prompt strict.

In [11]:
case = test_cases[0]

prompt = build_system_prompt_v2(
    rules=rules,
    fiche=case["fiche"],
)

responses = []

for _ in range(3):

    agent = BaselineAgent(
        model=llm.model,
        tokenizer=llm.tokenizer,
        device=llm.device,
        rules=rules,
        fiche=case["fiche"],
        system_prompt=prompt,
    )

    result = agent.ask(case["question"])

    responses.append(result["response"])

for i, response in enumerate(responses, start=1):
    print(f"\n--- Exécution {i} ---")
    print(response)


--- Exécution 1 ---
[VERDICT: ADMISSIBLE]

--- Exécution 2 ---
[VERDICT: ADMISSIBLE]

--- Exécution 3 ---
[VERDICT: ADMISSIBLE]


# Vérification de stabilité

Contrôle si les réponses sont identiques.

In [12]:
stable = len(set(responses)) == 1

print("Réponse stable :", stable)

Réponse stable : True


# Exactitude sémantique

Sépare la qualité du verdict du respect du format.

In [13]:
semantic_correct_v1 = {
    1: True,
    2: True,
    3: True,
    4: False,
    5: False,
    6: False,
}

df["semantic_correct"] = df["correct"]

for case_id, value in semantic_correct_v1.items():
    mask = (
        (df["prompt"] == "V1_simple")
        & (df["case_id"] == case_id)
    )

    df.loc[mask, "semantic_correct"] = value

semantic_summary = (
    df.groupby("prompt")
    .agg(
        semantic_accuracy=("semantic_correct", "mean"),
        format_rate=("format_valid", "mean"),
    )
    * 100
)

semantic_summary

,semantic_accuracy,format_rate
prompt,,
V1_simple,50.0,0.0
V2_strict,50.0,100.0
